# The full benchmark — 10 models + HAR-RV × 10 datasets × h = 1, 5, 22

Trains every deep model on all five forex and all five crypto realized-variance
series at horizons 1, 5 and 22, fits HAR-RV on the same rows, and writes

* the metric tables — MSE, MAE and QLIKE, per dataset, horizon, model and seed;
* the **per-observation loss series** that Diebold–Mariano and the MCS need.

Hyper-parameters come from the Optuna winners in `tuning/ProjectC_tuning`.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

**This does not finish in one session.** 300 cells × 10 seeds = **3 000
trainings**; a Colab session is capped at ~12 h. Everything is written to
Google Drive and the sweep **resumes** — reconnect, re-run steps 2–4, then run
step 7 again, and the cells already on Drive are skipped. Step 8 shows how far
along you are; step 9's tables work on a half-finished sweep.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU — Runtime > Change runtime type > T4 GPU. 3000 trainings on CPU is not realistic.'

## 2. Mount Drive

The forecasts, tables and loss matrices go here so they survive a disconnect —
that is what makes the sweep resumable across sessions.

Checkpoints do **not** go to Drive: they are rewritten every improving epoch,
and on a Drive mount that would dominate the runtime. They are deleted after
each cell anyway.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/ProjectC_benchmark'   # <- change if you like
CKPT_DIR    = '/content/_ckpt'                              # local disk, deleted per cell

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results ->', RESULTS_DIR)

## 3. Clone the repository

`Mr0022/ProjectC` is public, so this needs no credentials. Re-running the cell
in a later session updates an existing clone instead of failing.

Switch `BRANCH` to `'main'` once this work is merged.

In [ ]:
import os, subprocess

BRANCH   = 'claude/orchestrate-model-eval-forex-crypto-rir1dh'
REPO_URL = 'https://github.com/Mr0022/ProjectC.git'
REPO_DIR = '/content/ProjectC'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)   # the code resolves models/ and data/ relatively
print(subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install the dependencies Colab is missing

Colab already ships torch, numpy, pandas, scikit-learn, statsmodels and
matplotlib. These are the extras this repo needs — `fast_pytorch_kmeans` for
AdaWaveNet, `reformer-pytorch`/`local-attention` because
`layers/SelfAttention_Family.py` imports them at module load.

In [ ]:
!pip install -q einops PyWavelets fast_pytorch_kmeans reformer-pytorch local-attention psutil

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 5. Validate every configuration (~1 minute, CPU)

Builds all 10 models at all 3 horizons and pushes one batch through each, then
prints how many forecasts each dataset holds per horizon. A configuration the
architecture rejects should surface here, not at hour six of the sweep.

In [ ]:
!python orchestrate/run_benchmark.py --validate

## 6. Smoke test (~1 minute)

Two cheap models on one dataset at one horizon, one seed, 5 epochs — written to
local disk, so a smoke cell can never be mistaken for a real one by the resume
logic. If this prints a metrics table, the environment is wired up correctly.

In [ ]:
!python -u orchestrate/run_benchmark.py \
    --datasets EURUSD --horizons 1 --models DLinear FITS HAR-RV --itr 1 --quick \
    --results_dir /content/_smoke --checkpoint_dir $CKPT_DIR 2>&1 | tail -20

## 7. The sweep

3 000 trainings (300 cells × 10 seeds) plus 10 HAR-RV fits. One subprocess per
cell, so an OOM or a CUDA fault costs one cell rather than the run; each cell
writes its forecasts the moment it finishes, and cells already on Drive are
skipped — which is what makes this cell safe to re-run after every disconnect.

Ordered dataset → horizon → model, so an interrupted run leaves **complete**
(dataset, horizon) blocks behind — the unit the MCS is defined over.

`EXTRA` runs a subset, which is the practical way to do this over several
sessions:

* `['--assets', 'crypto']` or `['--datasets', 'EURUSD', 'AUDUSD']`
* `['--horizons', '1']`
* `['--itr', '3']` — 3 repeats per cell instead of 10, roughly a third of the cost
* `['--models', 'FITS', 'DLinear', 'TSLANet']` — the cheap ones first

In [ ]:
import subprocess, time

EXTRA = []          # e.g. ['--assets', 'crypto'] or ['--itr', '3']

started = time.time()
subprocess.run(['python', '-u', 'orchestrate/run_benchmark.py',
                '--results_dir', RESULTS_DIR,
                '--checkpoint_dir', CKPT_DIR] + EXTRA)
print(f'\nthis session ran for {(time.time() - started) / 3600:.2f} h')

## 8. How far along is it?

The planner is the authority on what is left — it counts the files on Drive the
same way the sweep does. The grid below shows completed cells per dataset and
horizon (110 = 11 models × 10 seeds, with HAR-RV counting once per horizon).

In [ ]:
!python orchestrate/run_benchmark.py --results_dir $RESULTS_DIR --dry_run 2>&1 | head -12

In [ ]:
import glob, os
import pandas as pd

paths = glob.glob(os.path.join(RESULTS_DIR, 'runs', '*', 'h*', '*.npz'))
if not paths:
    print('nothing on Drive yet')
else:
    done = pd.DataFrame([{'dataset': p.split(os.sep)[-3],
                          'horizon': p.split(os.sep)[-2]} for p in paths])
    print(f'{len(done)} cell(s) on Drive')
    display(done.value_counts().unstack(fill_value=0))

## 9. The tables

`aggregate_results.py` re-scores whatever is on Drive — no retraining — so this
also works on a sweep that is still running. It says which blocks are still
short of models.

In [ ]:
!python orchestrate/aggregate_results.py --results_dir $RESULTS_DIR

In [ ]:
import os
import pandas as pd

TABLES = os.path.join(RESULTS_DIR, 'tables')

# mean +/- std over the seeds, per dataset and horizon
display(pd.read_csv(os.path.join(TABLES, 'metrics_mean.csv')).head(20))

# models x datasets, one metric, one horizon -- the shape a paper table has
for h in (1, 5, 22):
    path = os.path.join(TABLES, f'pivot_QLIKE_h{h:02d}.csv')
    if os.path.exists(path):
        print(f'\nQLIKE, h = {h}')
        display(pd.read_csv(path, index_col=0).round(6))

## 10. The DM / MCS inputs

`losses/<dataset>_h<hh>__<loss>__seed<S>.csv` is date-indexed, one column per
model, one row per forecast, for each of `se_ln`, `ae_ln`, `qlike`, `se_rv`,
`ae_rv`. The mean of a column **is** the matching cell in `metrics.csv`.

**Run the tests on the `__seedmean` files**: one series per model — the forecast
its ten repeats average to — so you get one DM statistic and one MCS, rather
than ten that cannot be pooled.

At h > 1 the target windows of consecutive rows overlap by h−1 days, so the
loss differential is autocorrelated *by construction*: the DM long-run variance
needs a HAC estimator with at least h−1 lags, and the MCS block bootstrap needs
a block length that respects the same overlap.

In [ ]:
import glob, os
import pandas as pd

LOSSES = os.path.join(RESULTS_DIR, 'losses')
pick = sorted(glob.glob(os.path.join(LOSSES, '*__qlike__seedmean.csv')))
if not pick:
    print('no seedmean matrices yet — they appear once a block has >1 seed')
else:
    path = pick[0]
    print(os.path.basename(path))
    L = pd.read_csv(path, index_col=0, parse_dates=True)
    print(L.shape, '(forecasts x models)')
    display(L.head())

    # HAR-RV minus each model, row by row: positive = the model loses less.
    # rsub with axis=0 broadcasts down the dates; a plain Series - DataFrame
    # would align on the column labels and give an all-NaN frame.
    d = L.drop(columns='HAR-RV').rsub(L['HAR-RV'], axis=0)
    display(d.mean().sort_values(ascending=False)
             .to_frame('mean QLIKE saved vs HAR-RV'))

## 11. Download the results (optional)

They are already on Drive. This is for pulling the small files — the tables and
the loss matrices, not the 3 000 `.npz` — onto your laptop in one archive.

In [ ]:
import os, zipfile

out = '/content/ProjectC_benchmark_tables.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in ('tables', 'forecasts', 'losses'):
        for root, _, names in os.walk(os.path.join(RESULTS_DIR, sub)):
            for name in names:
                path = os.path.join(root, name)
                z.write(path, os.path.relpath(path, RESULTS_DIR))
    manifest = os.path.join(RESULTS_DIR, 'manifest.json')
    if os.path.exists(manifest):
        z.write(manifest, 'manifest.json')

print(out, round(os.path.getsize(out) / 1e6, 1), 'MB')

from google.colab import files
files.download(out)


## Notes

* **Resuming** — a cell whose `.npz` is on Drive is skipped, so re-running step
  7 continues where it stopped. A cell that failed is recorded in
  `failures.csv` and skipped too; `['--retry_failed']` in `EXTRA` re-runs those.
* **Drive is slow for many small files.** The sweep writes one `.npz` per cell
  (3 000 of them) and the aggregation a few thousand CSVs. That is fine, but a
  step-9 run takes a minute or two once the grid is full.
* **Hyper-parameters** are the Optuna winners for **EUR/USD at h = 1**, applied
  to every dataset and horizon. That is a transfer, not a per-cell search — it
  is the fair-comparison choice, and it belongs in the caption of any table
  built from these results. Per-dataset anchors are a drop-in: put
  `tuning/ProjectC_tuning/<dataset>/<Model>_best.json` in place.
* **What is scored** — the h-day forward mean of RV, which is HAR-RV's target
  `Y^(h)`. The scale comes from the anchors: they carry `--log` today, so the
  sweep is `ln_RV`; point `--anchor_dir` at a study tuned without it and the
  whole pipeline goes raw. Every model in a block is scored against the same
  actuals, rebuilt from the CSV in float64, and each cell's deviation from them
  is reported as `target_dev` in `metrics.csv`.
* **Cost** — FITS, DLinear and TSLANet are seconds per cell; TimesNet and
  MSGNet dominate. Plan on several sessions for the full 3 000, or start with
  `['--itr', '3']`.
* Details, file formats and the two `exp/` changes this work made:
  `orchestrate/README.md`.